# Примеры решения универсальной задачи "удаления избыточночных терминов из заданного списка без потери информации в рамках заданной перспективы"



**Для запуска вычислений, в секретах (Keys) этого .ipynb ноутбука необходимо задать значения следующих переменных:**
- `OPENAI_BASE_URL` (пример: `https://api.deepseek.com`)
- `OPENAI_API_KEY` (пример: `sk-...`)
- `OPENAI_MODEL` (пример: `deepseek-chat`)


## Настройка среды, вспомогательных функций и общих переменных для вычисления

Установка зависимостей и пакета [core-kbt](https://https://github.com/ady1981/core-kbt):

In [1]:
#rm -r /content/processes ## Delete process cache

In [2]:
!pip install -q poetry
!poetry config virtualenvs.in-project false
!rm -r /content/core-kbt
%cd /content
!git clone https://github.com/ady1981/core-kbt
%cd /content/core-kbt
!mv -n /content/core-kbt/processes /content/processes ## Keep cache of process results separately in /content/processes
!rm -r /content/core-kbt/processes
!ln -s /content/processes /content/core-kbt/processes ## Link to the cache
!poetry install

/content
Cloning into 'core-kbt'...
remote: Enumerating objects: 1798, done.
remote: Counting objects: 100% (309/309), done.
remote: Compressing objects: 100% (203/203), done.
remote: Total 1798 (delta 199), reused 205 (delta 105), pack-reused 1489 (from 1)
Receiving objects: 100% (1798/1798), 445.70 KiB | 13.93 MiB/s, done.
Resolving deltas: 100% (1090/1090), done.
/content/core-kbt
Installing dependencies from lock file

No dependencies to install or update

Installing the current project: kbt_core (0.2.11)Installing the current project: kbt_core (0.2.11)


In [3]:
# extra hack
!pip install ruamel.yaml
!pip install clorm

Установка переменных среды для openAI-compatible API:

In [4]:
import os
from google.colab import userdata
os.environ['OPENAI_BASE_URL'] = userdata.get('OPENAI_BASE_URL')
os.environ['OPENAI_MODEL'] = userdata.get('OPENAI_MODEL') ## Название LLM модели
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

Вспомогательные функции для вычисления:

In [45]:
import asyncio
import os
import pandas as pd
from google.colab import data_table

from kbt_core.ai_function import evaluate_function
from kbt_core.common import with_model_input_data, get_float, log_str
from kbt_core.common import index_by, with_only_keys
from kbt_core.process import read_ids, update_status

data_table.enable_dataframe_formatter()

def update_processes_to_initial():
    for input_id in read_ids('running') + read_ids('error'):
        log_str(f'--- update-process-to-initial: input_id={input_id}')
        update_status(input_id, 'initial')

async def calc_concept_set_covering(observer_context_description, concepts):
    frame_of_reference = 'Unbiased logical framework'
    input_data = {
        'observer_context_description': observer_context_description,
        'concepts': concepts,
        'frame_of_reference': frame_of_reference
    }
    return await evaluate_function('concept_set_covering', with_model_input_data(input_data, os.environ['OPENAI_MODEL']))

def is_deleted(concept, result):
  return not (concept in [c['concept'] for c in result['covering_concepts']])

def str_relation(relation):
  return f"{relation.get('schema', '').removesuffix('_schema')}: {','.join([c for c in relation.get('range', [])])}"

def is_same_concepts(a_concept, b_concept, result):
  concept_index = index_by(lambda c: c['concept'], result['covering_concepts']) | index_by(lambda c: c['concept'], result['eliminated_concepts'])
  return bool(concept_index.get(a_concept, {})['by_range_relations'].get(b_concept, {}).get('sameIdentityAs_schema', False)) or \
    bool(concept_index.get(b_concept, {})['by_range_relations'].get(a_concept, {}).get('sameIdentityAs_schema', False))

def create_output_tables(result, observer_context_description, concepts):
  llm_model = os.environ['OPENAI_MODEL']
  total_cols = {
      'result_name': ['input_terms', 'result_terms', 'deleted_terms', 'LLM_model', 'observer_context_description', 'perspective / basis_of_consideration', 'perspective / perspective_observer_strategy', 'perspective / point_of_view'],
      'result_value':
        [
            ', '.join(concepts),
            ', '.join([c['concept'] for c in result['covering_concepts']]),
            ', '.join([c['concept'] for c in result['eliminated_concepts']]),
            llm_model,
            observer_context_description,
            result.get('perspective', {}).get('basis_of_consideration'),
            result.get('perspective', {}).get('perspective_observer_strategy'),
            result.get('perspective', {}).get('point_of_view')
          ]
      }
  result_concepts = [c['concept'] for c in result['covering_concepts']]
  eliminated_concepts = index_by(lambda c: c['concept'], result['eliminated_concepts'])
  deleted_cols = {
      'input_term': [c for c in concepts if is_deleted(c, result)],
      'relations': ['; '.join(
          set([str_relation(list(c.values())[0]) for c in eliminated_concepts[concept]['by_range_relations'].values()] +
            [f'sameIdentityAs: {c}' for c in result_concepts if is_same_concepts(concept, c, result)])
          )
        for concept in concepts if is_deleted(concept, result)]
      }
  total_dt = pd.DataFrame(total_cols)
  deleted_dt = pd.DataFrame(deleted_cols)
  return (total_dt, deleted_dt)

## update process cache to initial state
update_processes_to_initial()


## Пример №1: Удаление избыточных терминов по биологической классификации

In [13]:
d01_observer_context_description = 'Биологическая классификация. Существенные признаки: принадлежность к классу'
d01_concepts = ['Растения', 'Животные', 'Человек']

d01_result = await calc_concept_set_covering(d01_observer_context_description, d01_concepts)
(d01_total_dt, d01_details_dt) = create_output_tables(d01_result, d01_observer_context_description, d01_concepts)


start: input_id=full_perspective_identification_af.1.3f8914212a77e71bfe150190c5b8514c.c37aae121f6b684f8f7431a68dcdfd92
process:
{
  "observer_context_description": "Биологическая классификация. Существенные признаки: принадлежность к классу",
  "frame_of_reference": "Unbiased logical framework",
  "_extra_output_specification": "# Extra output specification\nOutput_content_language: English",
  "meta": {
    "model": "deepseek/deepseek-pro"
  },
  "process_type": "full_perspective_identification_af",
  "input_id": "full_perspective_identification_af.1.3f8914212a77e71bfe150190c5b8514c.c37aae121f6b684f8f7431a68dcdfd92"
}
--- instruction meta: {
  "meta": {
    "model": "deepseek/deepseek-pro"
  }
}
--- instruction:
Generate items strictly satisfying to:
* Target specification
* Information retrieval strategy
* Output generation strategy
* Output specification.

# Target specification
Task_description: to identify full perspective representation for given Frame of reference and Observer c

In [14]:
d01_total_dt

,result_name,result_value
0,input_terms,"Растения, Животные, Человек"
1,result_terms,"Растения, Животные"
2,deleted_terms,Человек
3,LLM_model,deepseek/deepseek-pro
4,observer_context_description,Биологическая классификация. Существенные приз...
5,perspective / basis_of_consideration,Biological classification with essential featu...
6,perspective / perspective_observer_strategy,Identify the full perspective representation b...
7,perspective / point_of_view,The observer focuses on the class level of bio...


In [15]:
d01_details_dt

,input_term,relations
0,Человек,subclassOf: Животные


## Пример №2: Выбор верхнеуровных терминов для каталога компьютерных комплектующих

In [27]:
d02_observer_context_description = 'Каталог компьютерных комплектующих. Учитывать, что все компоненты поставляются в несобранном виде. Существенные признаки: принадлежность к подтипу, игнорировать функциональные отношения'
d02_concepts = 'CPU, RAM, hard drive, motherboard, video card, SSD, NVMe, Nvidia GeForce RTX 50'.split(', ')

d02_result = await calc_concept_set_covering(d02_observer_context_description, d02_concepts)
(d02_total_dt, d02_details_dt) = create_output_tables(d02_result, d02_observer_context_description, d02_concepts)

execute_process: already terminated, input_id=full_perspective_identification_af.1.3f8914212a77e71bfe150190c5b8514c.30d76e57a865633f151c4d16b2e4e0da
execute_process: already terminated, input_id=perspective_concept_relations_af.6.3f8914212a77e71bfe150190c5b8514c.17049be389dda98eb0b923174f3b4c46
execute_process: already terminated, input_id=perspective_concept_relations_af.6.3f8914212a77e71bfe150190c5b8514c.e7504d333d151dbae8c4a70c7c049ad4
execute_process: already terminated, input_id=perspective_concept_relations_af.6.3f8914212a77e71bfe150190c5b8514c.0fa6fbd8671d57eb7fb104c18506fedb
execute_process: already terminated, input_id=perspective_concept_relations_af.6.3f8914212a77e71bfe150190c5b8514c.304e15d32516ecff85513eeaa25d608b
execute_process: already terminated, input_id=perspective_concept_relations_af.6.3f8914212a77e71bfe150190c5b8514c.23116f6384abc7741783b715ec8f20b5
execute_process: already terminated, input_id=perspective_concept_relations_af.6.3f8914212a77e71bfe150190c5b8514c.a9

In [28]:
d02_total_dt

,result_name,result_value
0,input_terms,"CPU, RAM, hard drive, motherboard, video card,..."
1,result_terms,"CPU, RAM, hard drive, motherboard, video card"
2,deleted_terms,"SSD, NVMe, Nvidia GeForce RTX 50"
3,LLM_model,deepseek/deepseek-pro
4,observer_context_description,"Каталог компьютерных комплектующих. Учитывать,..."
5,perspective / basis_of_consideration,"Catalog of computer components, all supplied u..."
6,perspective / perspective_observer_strategy,Identify full perspective representation by cl...
7,perspective / point_of_view,Taxonomic classification of unassembled comput...


In [29]:
d02_details_dt

,input_term,relations
0,SSD,subclassOf: hard drive
1,NVMe,subclassOf: SSD
2,Nvidia GeForce RTX 50,subclassOf: video card


## Пример №3: Удаление избыточных терминов в описании этапов разработки программного обеспечения

In [19]:
d03_observer_context_description = 'Описание этапов разработки программного обеспечения для заказчика'
d03_concepts = 'Анализ требований, Кодирование, Тестирование, Развертывание, Приемочное тестирование, Фаза подготовки требований, Фаза исполнения, Фаза завершения'.split(', ')

d03_result = await calc_concept_set_covering(d03_observer_context_description, d03_concepts)
(d03_total_dt, d03_details_dt) = create_output_tables(d03_result, d03_observer_context_description, d03_concepts)

execute_process: already terminated, input_id=full_perspective_identification_af.1.3f8914212a77e71bfe150190c5b8514c.8afb8712e38310b2a3804fb113db64c0
start: input_id=perspective_concept_relations_af.6.3f8914212a77e71bfe150190c5b8514c.dcba596a57e9f5cb0381f09affbb08ac
process:
{
  "a_concept": "```yaml\n\n- name: Анализ требований\n  kind: entity\n  schema: entity_schema\n\n```\n",
  "b_concepts": "```yaml\n\n- name: Кодирование\n  kind: entity\n  schema: entity_schema\n\n- name: Тестирование\n  kind: entity\n  schema: entity_schema\n\n- name: Развертывание\n  kind: entity\n  schema: entity_schema\n\n- name: Приемочное тестирование\n  kind: entity\n  schema: entity_schema\n\n- name: Фаза подготовки требований\n  kind: entity\n  schema: entity_schema\n\n- name: Фаза исполнения\n  kind: entity\n  schema: entity_schema\n\n- name: Фаза завершения\n  kind: entity\n  schema: entity_schema\n\n```\n",
  "ontology_schema": "```yaml\n- name: item_schema\n  description: |\n    A root item schema.\n 

In [20]:
d03_total_dt

,result_name,result_value
0,input_terms,"Анализ требований, Кодирование, Тестирование, ..."
1,result_terms,"Фаза подготовки требований, Фаза исполнения, Ф..."
2,deleted_terms,"Анализ требований, Кодирование, Тестирование, ..."
3,LLM_model,deepseek/deepseek-pro
4,observer_context_description,Описание этапов разработки программного обеспе...
5,perspective / basis_of_consideration,Описание этапов разработки программного обеспе...
6,perspective / perspective_observer_strategy,Identify full perspective representation by an...
7,perspective / point_of_view,Customer-centric view of the software developm...


In [21]:
d03_details_dt

,input_term,relations
0,Анализ требований,partOf: Фаза подготовки требований
1,Кодирование,partOf: Фаза исполнения
2,Тестирование,partOf: Фаза исполнения
3,Развертывание,partOf: Фаза завершения
4,Приемочное тестирование,subclassOf: Тестирование; partOf: Фаза завершения


## Пример №4: Создание глоссария для внутреннего руководства по разработке программного обеспечения

In [46]:
d04_observer_context_description = 'Создание глоссария для внутреннего руководства по разработке программного обеспечения с перспективы разработчика. Существенные признаки: принадлежность к подтипу, игнорировать функциональные отношения'
d04_concepts = '''Пользовательский интерфейс
Интерфейс пользователя
Фронтенд
Серверная часть
Бэкенд
База данных
Хранилище данных'''.split('\n')

d04_result = await calc_concept_set_covering(d04_observer_context_description, d04_concepts)
(d04_total_dt, d04_details_dt) = create_output_tables(d04_result, d04_observer_context_description, d04_concepts)

execute_process: already terminated, input_id=full_perspective_identification_af.1.3f8914212a77e71bfe150190c5b8514c.517e17db06891e22a05980e966a0d1ad
execute_process: already terminated, input_id=perspective_concept_relations_af.6.3f8914212a77e71bfe150190c5b8514c.2aa71a0b05b6c6f7dc10e59ed1ef840b
execute_process: already terminated, input_id=perspective_concept_relations_af.6.3f8914212a77e71bfe150190c5b8514c.53654587f093caa717015ad61ec563b1
execute_process: already terminated, input_id=perspective_concept_relations_af.6.3f8914212a77e71bfe150190c5b8514c.ef48b02b91648ef073ecaba4261511ca
execute_process: already terminated, input_id=perspective_concept_relations_af.6.3f8914212a77e71bfe150190c5b8514c.a363f933c5c2d81b9fdb9c28c6fa3191
execute_process: already terminated, input_id=perspective_concept_relations_af.6.3f8914212a77e71bfe150190c5b8514c.8dc3078d7134b953b2a9337c6bc5d13d
execute_process: already terminated, input_id=perspective_concept_relations_af.6.3f8914212a77e71bfe150190c5b8514c.03

In [47]:
d04_total_dt

,result_name,result_value
0,input_terms,"Пользовательский интерфейс, Интерфейс пользова..."
1,result_terms,"Фронтенд, Бэкенд, База данных"
2,deleted_terms,"Пользовательский интерфейс, Интерфейс пользова..."
3,LLM_model,deepseek/deepseek-pro
4,observer_context_description,Создание глоссария для внутреннего руководства...
5,perspective / basis_of_consideration,"Subtype membership, ignoring functional relati..."
6,perspective / perspective_observer_strategy,Developer-centric glossary creation for intern...
7,perspective / point_of_view,Developer


In [48]:
d04_details_dt

,input_term,relations
0,Пользовательский интерфейс,sameIdentityAs: Интерфейс пользователя; sameId...
1,Интерфейс пользователя,sameIdentityAs: Пользовательский интерфейс; sa...
2,Серверная часть,sameIdentityAs: Бэкенд
3,Хранилище данных,sameIdentityAs: База данных


## Пример №5: Оптимизация набора ключевых тем для статьи


In [49]:
d05_observer_context_description = 'Мы готовим обзорную статью по теме "Использование машинного обучения для улучшения пользовательского опыта в электронной коммерции". Нам необходимо составить максимально лаконичный набор ключевых тем/идей для этой статьи.'
d05_concepts = '''Машинное обучение
Персонализация
Электронная коммерция
Рекомендательные системы
Алгоритмы ранжирования товаров
Искусственный интеллект
Поведенческий анализ пользователей
Увеличение конверсии
Динамическое ценообразование на основе ML
Прогнозирование оттока клиентов
Индивидуальные предложения
Оптимизация пользовательского пути
Глубокое обучение
Сегментация клиентов'''.split('\n')

d05_result = await calc_concept_set_covering(d05_observer_context_description, d05_concepts)
(d05_total_dt, d05_details_dt) = create_output_tables(d05_result, d05_observer_context_description, d05_concepts)

execute_process: already terminated, input_id=full_perspective_identification_af.1.3f8914212a77e71bfe150190c5b8514c.449850ab6cef044a70382a79464ce2e4
execute_process: already terminated, input_id=perspective_concept_relations_af.6.3f8914212a77e71bfe150190c5b8514c.ca06d3e48dd24ec2785c0fe716a4f0b9
execute_process: already terminated, input_id=perspective_concept_relations_af.6.3f8914212a77e71bfe150190c5b8514c.649d0c50ff878cd7bf7b36215b24a2f1
execute_process: already terminated, input_id=perspective_concept_relations_af.6.3f8914212a77e71bfe150190c5b8514c.90b145131c74285c2ed5ef8877e9ed7b
execute_process: already terminated, input_id=perspective_concept_relations_af.6.3f8914212a77e71bfe150190c5b8514c.c881a79f06f5bcf2a40f60c025c2f5fe
execute_process: already terminated, input_id=perspective_concept_relations_af.6.3f8914212a77e71bfe150190c5b8514c.cf3ed19fff1edf4f09345e025ee1ca36
execute_process: already terminated, input_id=perspective_concept_relations_af.6.3f8914212a77e71bfe150190c5b8514c.67

In [50]:
d05_total_dt

,result_name,result_value
0,input_terms,"Машинное обучение, Персонализация, Электронная..."
1,result_terms,"Электронная коммерция, Искусственный интеллект..."
2,deleted_terms,"Машинное обучение, Персонализация, Рекомендате..."
3,LLM_model,deepseek/deepseek-pro
4,observer_context_description,"Мы готовим обзорную статью по теме ""Использова..."
5,perspective / basis_of_consideration,The task is to identify a full perspective rep...
6,perspective / perspective_observer_strategy,Extract core themes from the intersection of m...
7,perspective / point_of_view,A comprehensive yet concise set of key topics ...


In [51]:
d05_details_dt

,input_term,relations
0,Машинное обучение,subclassOf: Искусственный интеллект
1,Персонализация,relation: Рекомендательные системы; relation: ...
2,Рекомендательные системы,relation: Поведенческий анализ пользователей; ...
3,Алгоритмы ранжирования товаров,partOf: Персонализация; subclassOf: Рекомендат...
4,Поведенческий анализ пользователей,partOf: Персонализация; relation: Рекомендател...
5,Динамическое ценообразование на основе ML,partOf: Поведенческий анализ пользователей; pa...
6,Прогнозирование оттока клиентов,partOf: Персонализация; relation: Рекомендател...
7,Индивидуальные предложения,relation: Рекомендательные системы; relation: ...
8,Оптимизация пользовательского пути,partOf: Увеличение конверсии; partOf: Электрон...
9,Глубокое обучение,partOf: Рекомендательные системы; subclassOf: ...
